In [5]:
import pandas as pd
from datetime import timedelta

In [6]:
selected_student = 80

context = pd.read_excel('context/Assessment_Information.xlsx')
context = context[context['student_id'] == selected_student]

context.head()

,student_id,question_id,answer,date,duration,option_selected,Topic,Subtopic,Lecturer_level,Algorithm_level,Typology
17046,80,142,1,2024-01-15T16:37:50.000Z,NaN,NaN,Differentiation,Derivatives,1,1,Admin
17047,80,94,1,2024-01-15T16:37:50.000Z,NaN,NaN,Differentiation,Derivatives,2,1,Admin
17048,80,734,0,2024-01-15T16:37:50.000Z,NaN,NaN,Differentiation,Derivatives,3,4,Admin
17049,80,274,1,2024-01-15T16:37:50.000Z,NaN,NaN,Differentiation,Derivatives,2,1,Admin
17050,80,721,0,2024-01-15T16:37:50.000Z,NaN,NaN,Differentiation,Derivatives,3,4,Admin


In [7]:
context['date'] = pd.to_datetime(context['date'])
last_interaction = context['date'].max()
context['days_since_last_interaction'] = (last_interaction - context['date']).dt.days

step_1 = timedelta(days=15)
step_2 = timedelta(days=30)
step_3 = timedelta(days=60)

# Assign lapse scores based on the days since last interaction: 1 for <= 15 days, 0.6 for 16-30 days, 0.3 for 31-60 days, and 0.1 for > 60 days.
context['lapse_score'] = pd.cut(context['days_since_last_interaction'],
                                   bins=[-1, step_1.days, step_2.days, step_3.days, float('inf')],
                                   labels=['1', '0.6', '0.3', '0.1'])

# sum(difficulty * lapse_score * answer)
context['knowledge_score'] = context['Algorithm_level'] * context['lapse_score'].astype(float) * context['answer']

display(context)



,student_id,question_id,answer,date,duration,option_selected,Topic,Subtopic,Lecturer_level,Algorithm_level,Typology,days_since_last_interaction,lapse_score,knowledge_score
17046,80,142,1,2024-01-15 16:37:50+00:00,NaN,NaN,Differentiation,Derivatives,1,1,Admin,364,0.1,0.1
17047,80,94,1,2024-01-15 16:37:50+00:00,NaN,NaN,Differentiation,Derivatives,2,1,Admin,364,0.1,0.1
17048,80,734,0,2024-01-15 16:37:50+00:00,NaN,NaN,Differentiation,Derivatives,3,4,Admin,364,0.1,0.0
17049,80,274,1,2024-01-15 16:37:50+00:00,NaN,NaN,Differentiation,Derivatives,2,1,Admin,364,0.1,0.1
17050,80,721,0,2024-01-15 16:37:50+00:00,NaN,NaN,Differentiation,Derivatives,3,4,Admin,364,0.1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19766,80,575,0,2025-01-14 15:58:14+00:00,NaN,NaN,Analytic Geometry,NaN,5,1,Admin,0,1,0.0
19767,80,1742,0,2025-01-14 15:58:23+00:00,NaN,NaN,Analytic Geometry,NaN,5,1,Admin,0,1,0.0
19768,80,561,-1,2025-01-14 15:58:27+00:00,NaN,NaN,Analytic Geometry,NaN,5,1,Admin,0,1,-1.0
19769,80,1740,1,2025-01-14 15:58:31+00:00,NaN,NaN,Analytic Geometry,NaN,4,1,Admin,0,1,1.0


In [10]:
new_context = context[['Topic', 'Subtopic', 'knowledge_score']]
new_context.groupby(['Topic', 'Subtopic']).agg({'knowledge_score': 'sum'}).reset_index()

,Topic,Subtopic,knowledge_score
0,Differentiation,Derivatives,2.1
1,Differentiation,Implicit Differentiation and Chain Rule,0.8
2,Differentiation,Partial Differentiation,0.3
3,Discrete Mathematics,Recursivity,1.2
4,Discrete Mathematics,Set Theory,4.0
5,Fundamental Mathematics,"Algebraic expressions, Equations, and Inequali...",1.2
6,Fundamental Mathematics,Elementary Geometry,0.1
7,Integration,Double Integration,0.4
8,Integration,Integration Techniques,0.1
9,Integration,Surface Integrals,0.6
